# Support Triage Agent — TCK-1017 Walkthrough

Traces a single ticket, **TCK-1017** (a GDPR data-export request), through
every node of the LangGraph pipeline
(`ingest -> sentiment_policy_check -> rag_retrieve -> draft_answer -> route ->
confidence_recheck -> hitl_gate`), by reading its persisted `GraphState`.

TCK-1017 asks about GDPR data export and data retention -- a topic the
knowledge base (refund, subscription, account-access, troubleshooting docs)
doesn't actually cover. It is the reference example for the 'no policy
found -> escalate' behavior: retrieval only turns up loosely related,
low-similarity chunks, the drafted reply says plainly that the relevant
policy could not be verified, and `route_decision` routes the ticket to
**ESCALATE** with reason `low_groundedness` rather than letting a
low-confidence answer reach the customer.

In [1]:
import sys, json, sqlite3
from pathlib import Path

# Works whether Jupyter's cwd is notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config.settings import get_settings
from src.main import load_tickets

settings = get_settings()
tickets_by_id = {t.ticket_id: t for t in load_tickets(settings)}
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded {len(tickets_by_id)} tickets from {settings.tickets_path.relative_to(REPO_ROOT)}")

def load_result(ticket_id: str) -> dict:
    return json.loads((settings.results_dir / f"{ticket_id}.json").read_text(encoding="utf-8"))

def latest_reviewer_action(ticket_id: str) -> str | None:
    """The result JSON's reviewer_action is only a snapshot of hitl_gate's
    output the moment the graph run finished -- None whenever the ticket went
    into the async reviewer queue instead of being auto-approved. The reviews
    table in outputs/databases/triage.db gets updated in place once a human
    (or the CLI's --queue processor) actually reviews it, so pull the most
    recently updated row for this ticket from there instead."""
    conn = sqlite3.connect(settings.reviewer_db_path)
    try:
        row = conn.execute(
            "SELECT reviewer_action FROM reviews WHERE ticket_id = ? ORDER BY updated_at DESC LIMIT 1",
            (ticket_id,),
        ).fetchone()
    finally:
        conn.close()
    return row[0] if row else None


/Users/mahendarprakash/SupportTriageAgent/.supportvenv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Repo root: /Users/mahendarprakash/SupportTriageAgent
Loaded 22 tickets from data/synthetic_tickets.json


## Node-by-node breakdown

One helper that prints the persisted `GraphState` for TCK-1017 grouped by
the node that produced each field -- same grouping as the other walkthrough
notebooks and the pipeline order documented in `ARCHITECTURE.md` §4.

In [2]:
def explain(ticket_id: str) -> dict:
    ticket = tickets_by_id[ticket_id]
    state = load_result(ticket_id)

    print(f"##### ticket (input) #####")
    print(json.dumps(ticket.model_dump(), indent=2))

    print(f"\n##### ingest #####")
    print(f"conversation_context: {state.get('conversation_context')!r}")

    print(f"\n##### sentiment_policy_check #####")
    for k in ["abuse_detected", "sentiment", "detected_category", "requires_more_info", "missing_fields"]:
        print(f"{k}: {state.get(k)}")

    print(f"\n##### rag_retrieve #####")
    chunks = state.get("retrieved_chunks") or []
    print(f"retrieval_attempts: {state.get('retrieval_attempts')}")
    for c in chunks:
        print(f"  [{c['score']:.3f}] {c['source']}: {c['text'][:90].strip()}...")

    print(f"\n##### draft_answer #####")
    print(f"groundedness_score (retrieval-similarity): {state.get('groundedness_score')}")
    print(f"fabricated_citations: {state.get('fabricated_citations')}")
    print(f"draft_reply:\n{state.get('draft_reply')}")

    print(f"\n##### route #####")
    print(f"route_decision: {state.get('route_decision')}")
    print(f"route_reason:   {state.get('route_reason')}")

    if state.get("llm_groundedness_score") is not None:
        print(f"\n##### confidence_recheck (only reached for AUTO_RESOLVE) #####")
        print(f"llm_groundedness_score:  {state.get('llm_groundedness_score')}")
        print(f"llm_groundedness_passed: {state.get('llm_groundedness_passed')}")
        print(f"unsupported_claims:      {state.get('unsupported_claims')}")
    else:
        print(f"\n##### confidence_recheck #####")
        print("skipped -- only reached when route_decision == AUTO_RESOLVE")

    print(f"\n##### hitl_gate #####")
    print(f"reviewer_action: {latest_reviewer_action(ticket_id)}  (review_id={state.get('review_id')})")

    return state


## TCK-1017 — GDPR data export request, no matching KB policy

Actual route: **ESCALATE** (reason: `low_groundedness`). Retrieval only
surfaces account-access, refund, and troubleshooting chunks with low
similarity scores -- none of them actually address GDPR data export or
retention. `route_decision` catches this at rule 6 (see
`src/graph/nodes/route_decision.py`): either no chunks at all
(`no_policy_found`) or a top similarity score below
`settings.groundedness_threshold` (`low_groundedness`), and escalates
before the ticket ever reaches `confidence_recheck`.

In [3]:
state = explain("TCK-1017")

##### ticket (input) #####
{
  "ticket_id": "TCK-1017",
  "customer_id": "CUST-017",
  "subject": "GDPR data export request",
  "message": "I want a complete copy of all my personal data under GDPR and details of your data retention policy.",
  "conversation_history": [],
  "priority": "medium",
  "category": "other",
  "order_id": null,
  "account_id": null,
  "subscription_id": null,
  "days_since_purchase": null,
  "previous_refund_request_count": 0,
  "days_since_last_refund_request": null
}

##### ingest #####
conversation_context: '(no conversation history)'

##### sentiment_policy_check #####
abuse_detected: False
sentiment: neutral
detected_category: other
requires_more_info: False
missing_fields: []

##### rag_retrieve #####
retrieval_attempts: 1
  [0.172] account_access_faq.md: ## Required Information
Before troubleshooting account access, request one of the followin...
  [0.152] refund_policy.md: # Refund Policy

## Scope
This policy applies to purchases made through the com

## Drafted reply text

The LLM draft still gets written (for reviewer context) even though the
ticket is escalated -- it explicitly declines to state unverified policy
and recommends escalation, per the system prompt in
`src/agents/response_agent.py`.

In [4]:
print(state["draft_reply"])

Draft Reply:

Regarding your request for a complete copy of your personal data under GDPR, the provided context does not specify the process or details for handling such requests. Additionally, the data retention policy is not mentioned in the available documents [source: account_access_faq.md, source: refund_policy.md, source: troubleshooting_faq.md]. 

As the relevant policy could not be verified, we recommend escalating this matter to our support team for further assistance. They will be able to provide you with the necessary information and guidance on how to proceed with your GDPR data export request.
